some adjustments to be made
0. Make sure all the dimensions follow the main logic of the matrix
1. Add the parameters for the tensor dimensions for the attention block.
2. Add the masking matrix to the self-attention block
3. Double check that all the multiplications go as planned. I recieve the expected outcomes.

In [ ]:
import math
import torch
import torch.nn as nn

D = 6 # number of dimensions (for the embeddings as far as i am concerned)
I = 5 # number of input tokens (as far as i am concerned
N_ATT = 2

X = torch.rand((D, I)) # Dimensions are rows, and I-inputs are individual tokens

In [17]:
class SelfAttention_Block(nn.Module):    
    def __init__(self, num_att_heads: int, D_dimension: int):
        super().__init__()
        self.D = D_dimension
        # linear projections  W_*  ∈ ℝ[D × D]
        self.Q_w = nn.Parameter(torch.randn(self.D, self.D))
        self.K_w = nn.Parameter(torch.randn(self.D, self.D))
        self.V_w = nn.Parameter(torch.randn(self.D, self.D))

        self.Q_b = nn.Parameter(torch.randn(self.D, 1))
        self.K_b = nn.Parameter(torch.randn(self.D, 1))
        self.V_b = nn.Parameter(torch.randn(self.D, 1))

        self.softmax = nn.Softmax(dim=1) 

    def forward(self, X: torch.Tensor) -> torch.Tensor:
        # 1) linear projections
        Q = self.Q_w @ X + self.Q_b            # (D, I)
        K = self.K_w @ X + self.K_b
        V = self.V_w @ X + self.V_b

        # 2) scaled dot-product attention
        attn = (Q.t() @ K) / math.sqrt(self.D)         # (I, I)
        attn   = self.softmax(attn)                      # (I, I)

        # 3) weighted sum of values for each query token
        context = (attn @ V.t()).t()                       # (D, I)
        return context

In [18]:
block = SelfAttention_Block(1, D)
out   = block(X)           # forward

loss  = out.sum()          # scalar, any criterion ok
block.zero_grad()          # clear
loss.backward()            # backward pass

print(block.K_w.grad)   # leaf param grad
print(block.V_w.grad)     # non-leaf grad (only if you saved it)

tensor([[ 1.6727e-01,  3.1949e-01,  8.6100e-02,  4.9599e-02,  2.9113e-01,
         -2.4822e-01],
        [-1.4996e-01, -2.8161e-01, -7.6261e-02, -4.7988e-02, -2.5832e-01,
          2.1840e-01],
        [-7.5773e-02, -1.5774e-01, -4.0038e-02, -1.4593e-02, -1.3740e-01,
          1.2317e-01],
        [-1.3257e-03,  2.6467e-03, -3.2407e-04, -3.4839e-03, -1.7634e-04,
         -2.2902e-03],
        [-7.2809e-02, -1.5349e-01, -3.9342e-02, -1.2154e-02, -1.3370e-01,
          1.2014e-01],
        [ 1.2932e-01,  2.7932e-01,  6.9223e-02,  1.8732e-02,  2.3890e-01,
         -2.1859e-01]])
tensor([[1.6566, 3.5909, 2.7342, 2.3104, 3.9430, 2.0769],
        [1.6566, 3.5909, 2.7342, 2.3104, 3.9430, 2.0769],
        [1.6566, 3.5909, 2.7342, 2.3104, 3.9430, 2.0769],
        [1.6566, 3.5909, 2.7342, 2.3104, 3.9430, 2.0769],
        [1.6566, 3.5909, 2.7342, 2.3104, 3.9430, 2.0769],
        [1.6566, 3.5909, 2.7342, 2.3104, 3.9430, 2.0769]])


In [19]:
class MultiHead_SelfAttention_Block(nn.Module):    
    def __init__(self, num_att_heads: int, D_dimension: int):
        super().__init__()
        assert D_dimension % num_att_heads == 0, "D must be divisible by the number of heads"
        self.heads_n = num_att_heads
        self.D = D_dimension
        self.d_k = D_dimension // self.heads_n            # head size

        self.Q_w = nn.ParameterList()
        self.K_w = nn.ParameterList()
        self.V_w = nn.ParameterList()

        self.Q_b = nn.ParameterList()
        self.K_b = nn.ParameterList()
        self.V_b = nn.ParameterList()

        self.O_w = nn.Parameter(torch.randn(self.D, self.D))
        # linear projections  W_*  ∈ ℝ[d_k × D]
        for i in range(num_att_heads):
            self.Q_w.append(nn.Parameter(torch.randn(self.d_k, self.D)))
            self.K_w.append(nn.Parameter(torch.randn(self.d_k, self.D)))
            self.V_w.append(nn.Parameter(torch.randn(self.d_k, self.D)))

            self.Q_b.append(nn.Parameter(torch.randn(self.d_k, 1)))
            self.K_b.append(nn.Parameter(torch.randn(self.d_k, 1)))
            self.V_b.append(nn.Parameter(torch.randn(self.d_k, 1)))

        self.softmax = nn.Softmax(dim=1) 

    def forward(self, X: torch.Tensor) -> torch.Tensor:
        head_contexts = []

        for i in range(self.heads_n):
            # 1) linear projections
            Q = self.Q_w[i] @ X + self.Q_b[i]            # (d_k, I)
            K = self.K_w[i] @ X + self.K_b[i]
            V = self.V_w[i] @ X + self.V_b[i]

            # 2) scaled dot-product attention
            attn = (Q.t() @ K) / math.sqrt(self.d_k)         # (I, I)
            attn   = self.softmax(attn)                      # (I, I)

            # 3) weighted sum of values for each query token
            context = (attn @ V.t()).t()                       # (d_k, I)
            head_contexts.append(context)  
        
        context = torch.cat(head_contexts, dim=0)  # (D, I)
        output = self.O_w @ context  # (D, I)
        return output

In [22]:
block = MultiHead_SelfAttention_Block(N_ATT, D)
out   = block(X)           # forward

loss  = out.sum()          # scalar, any criterion ok
block.zero_grad()          # clear
loss.backward()            # backward pass

print(block.K_w[0].grad)   # leaf param grad
print(block.O_w.grad)     # non-leaf grad (only if you saved it)

tensor([[-0.0451, -0.1429, -0.0162,  0.0061, -0.0895,  0.1106],
        [-0.4464, -0.6079,  0.0260, -0.5595, -0.4006,  0.3958],
        [-0.2013, -0.3124, -0.0010, -0.2191, -0.2084,  0.2127]])
tensor([[ 0.7990,  2.4726,  3.9991, -1.2135, -4.3423, -5.0579],
        [ 0.7990,  2.4726,  3.9991, -1.2135, -4.3423, -5.0579],
        [ 0.7990,  2.4726,  3.9991, -1.2135, -4.3423, -5.0579],
        [ 0.7990,  2.4726,  3.9991, -1.2135, -4.3423, -5.0579],
        [ 0.7990,  2.4726,  3.9991, -1.2135, -4.3423, -5.0579],
        [ 0.7990,  2.4726,  3.9991, -1.2135, -4.3423, -5.0579]])
